# Notebook QA Checker

Checks every `.ipynb` lesson file for:
1. Has markdown cell with `══` border
2. No `input()` calls in code cells
3. No emojis
4. At least 3 EXERCISE/TASK sections (in markdown cells)
5. At least 3 EXAMPLE/PART blocks (non-project files only)
6. Has empty code cell (student workspace)
7. Code cells have valid Python syntax

In [1]:
import os
import ast
import re
import json

CURRICULUM_DIR = os.path.abspath(os.path.dirname("__file__"))
if not os.path.exists(os.path.join(CURRICULUM_DIR, "README.py")):
    CURRICULUM_DIR = os.path.abspath(".")

SKIP_FILES = {"verify_curriculum.ipynb", "qa.ipynb", "README.ipynb"}
EMOJI_PATTERN = re.compile(r"[\U0001F000-\U0010FFFF]")

In [2]:
def read_notebook(filepath):
    """Read a .ipynb file and return its parsed JSON."""
    with open(filepath, encoding="utf-8") as f:
        return json.load(f)


def get_cell_source(cell):
    """Return the full text of a cell's source."""
    src = cell.get("source", [])
    if isinstance(src, list):
        return "".join(src)
    return src


def check_notebook(filepath):
    """Return a list of issues found in a .ipynb lesson file."""
    issues = []

    try:
        nb = read_notebook(filepath)
    except Exception as e:
        return [f"could not read notebook: {e}"]

    cells = nb.get("cells", [])
    if not cells:
        return ["notebook has no cells"]

    # Separate cell types
    md_cells = [c for c in cells if c.get("cell_type") == "markdown"]
    code_cells = [c for c in cells if c.get("cell_type") == "code"]

    # Combine all text for full-file checks
    all_md_text = "\n".join(get_cell_source(c) for c in md_cells)
    all_code_text = "\n".join(get_cell_source(c) for c in code_cells)
    all_text = all_md_text + "\n" + all_code_text

    # 1. Border format
    if "\u2550\u2550" not in all_text:
        issues.append("missing border -- no markdown cell with border found")

    # 2. No input() in code cells
    if "input(" in all_code_text:
        issues.append("contains input() in code cells -- not allowed")

    # 3. No emojis
    if EMOJI_PATTERN.search(all_text):
        issues.append("contains emoji characters -- remove them")

    # 4. At least 3 EXERCISE or TASK markers in markdown
    exercise_count = all_md_text.count("EXERCISE") + all_md_text.count("TASK")
    if exercise_count < 3:
        issues.append(f"only {exercise_count} EXERCISE/TASK section(s) -- need at least 3")

    # 5. At least 3 EXAMPLE or PART markers (skip project/overview files)
    is_project = re.search(r"(_D6_|_D7_|OVERVIEW|_Project_|_project_)", filepath)
    if not is_project:
        example_count = all_md_text.count("EXAMPLE") + all_md_text.count("PART")
        if example_count < 3:
            issues.append(f"only {example_count} EXAMPLE/PART block(s) -- need at least 3")

    # 6. Student workspace: at least one empty/whitespace-only code cell
    has_workspace = any(
        get_cell_source(c).strip() == "" or
        re.match(r"^\s*$", get_cell_source(c))
        for c in code_cells
    )
    if not has_workspace and exercise_count >= 1:
        issues.append("no empty code cell found for student workspace")

    # 7. Python syntax in code cells
    for i, cell in enumerate(code_cells):
        src = get_cell_source(cell)
        if not src.strip():
            continue
        try:
            ast.parse(src)
        except SyntaxError as e:
            issues.append(f"SYNTAX ERROR in code cell {i + 1}, line {e.lineno}: {e.msg}")

    return issues

In [3]:
def collect_notebook_files():
    """Walk Curriculum/ and return sorted .ipynb lesson file paths."""
    nb_files = []
    for root, dirs, files in os.walk(CURRICULUM_DIR):
        dirs.sort()
        for filename in sorted(files):
            if not filename.endswith(".ipynb"):
                continue
            if filename in SKIP_FILES:
                continue
            nb_files.append(os.path.join(root, filename))
    return nb_files

## Run QA

In [5]:
nb_files = collect_notebook_files()
results = {}
for filepath in nb_files:
    rel = os.path.relpath(filepath, CURRICULUM_DIR)
    results[rel] = check_notebook(filepath)

total_files = len(results)
total_issues = sum(len(v) for v in results.values())
files_passing = sum(1 for v in results.values() if not v)
files_failing = total_files - files_passing

print("=" * 64)
print("  NOTEBOOK QA REPORT (.ipynb files)")
print(f"  Files checked : {total_files}")
print(f"  Passing       : {files_passing}")
print(f"  Failing       : {files_failing}")
print(f"  Total issues  : {total_issues}")
print("=" * 64)
print()

current_week = None
for rel_path, issues in results.items():
    week_folder = rel_path.split(os.sep)[0] if os.sep in rel_path else ""
    if week_folder != current_week:
        current_week = week_folder
        print(f"  {week_folder}")
        print(f"  {'---' * 17}")
    filename = os.path.basename(rel_path)
    status = "PASS" if not issues else "FAIL"
    print(f"    {status}  {filename}")
    for issue in issues:
        print(f"          - {issue}")

print()
print("=" * 64)
if files_failing == 0:
    print("  All notebooks pass. QA clean.")
else:
    print(f"  {files_failing} notebook(s) need attention. Fix issues above and re-run.")
print("=" * 64)

  NOTEBOOK QA REPORT (.ipynb files)
  Files checked : 78
  Passing       : 76
  Failing       : 2
  Total issues  : 4

  Week_10_AI_Engineering
  ---------------------------------------------------
    PASS  W10_D1_LLM_APIs.ipynb
    PASS  W10_D2_Prompt_Engineering.ipynb
    PASS  W10_D3_LangChain_Basics.ipynb
    PASS  W10_D4_VectorDB_Embeddings.ipynb
    PASS  W10_D5_Project_RAG_Pipeline.ipynb
    PASS  W10_D6_Project_RAG_Chatbot.ipynb
  Week_11_Advanced_AI
  ---------------------------------------------------
    PASS  W11_D1_LangGraph_Agents.ipynb
    PASS  W11_D2_NLP_Basics.ipynb
    PASS  W11_D3_Speech_Recognition.ipynb
    PASS  W11_D4_AI_Ethics.ipynb
    PASS  W11_D5_Final_AI_Project.ipynb
    PASS  W11_D6_Project_AI_Data_Assistant.ipynb
  Week_12_Capstone
  ---------------------------------------------------
    PASS  W12_D1_Docker_Kafka_Setup.ipynb
    PASS  W12_D2_Extract_API_Producer.ipynb
    PASS  W12_D3_Store_MinIO.ipynb
    PASS  W12_D4_Transform_Load_PostgreSQL.ipynb
 